In [ ]:
%pip install -q ipylab

# Autostart

Autostart is a feature implemented using the [`pluggy`](https://pluggy.readthedocs.io/en/stable/index.html#pluggy) plugin system. Plugins registered under the entry point `ipylab-python-backend` will be called once when `ipylab` is activated. Normally activation of `ipylab` will occur when Jupyterlab is started (assuming `ipylab` is installed and enabled). 

There are no limitations to what can be done. But some possibilities include:
* Launch an app to run in its own thread;
* Create and register custom commands;
* Create launchers;
* Create new notebooks;

## Entry points

Add the following in your `pyproject.toml`

``` toml
[project.entry-points.ipylab-python-backend]
my_plugins = "my_module.autostart"
```

In `my_module.autostart.py` define the plugins.

Example:

```python
# @autostart.py

import ipylab

app = ipylab.JupyterFrontEnd()

def create_app():
  Add code here to create the app

class MyPlugins:
    @ipylab.hookspecs.hookimpl()
    def run_once_at_startup(self):
        app.newSession(path="my app", code=create_app)
        # Do more stuff ...
```

## Example lauching a small app

In [ ]:
# @my_module.autostart.py

import ipylab

app = ipylab.JupyterFrontEnd()


async def create_app():
    # Ensure this function provides all the imports.
    global ma
    import ipywidgets as ipw

    import ipylab

    # app = ipylab.JupyterFrontEnd()
    # await app.wait_ready()
    ma = ipylab.MainArea(name="My demo app")
    console_button = ipw.Button(description="Toggle console")
    console_button.on_click(
        lambda b: ma.load_console() if not ma.console_loaded else ma.unload_console()
    )
    ma.content.children = [
        ipw.HTML(f"<h3>My simple app</h3> Welcome to my app.<br> kernel id: {ma.kernelId}"),
        console_button,
    ]
    ma.content.label = "This is my app"
    ma.load()
    print("Finshed creating my app")


class MyPlugins:
    @ipylab.hookspecs.hookimpl()
    def run_once_at_startup(self):
        app.newSession(path="my app", code=create_app)


MyPlugins()

### Launch the app manually

We can 'launch' the app in a new kernel.

In [ ]:
app.newSession(path="my app", code=create_app)

### Auto Lauch app
Simulate code launch in the as it happens in `Ipylab backend`

In [ ]:
# Register plugin (normally via the entry point `ipylab-python-backend`)
ipylab.hookspecs.pm.register(MyPlugins())

# Called when Ipylab is activated and Ipylab backend launches
app._init_python_backend()